In [4]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv



In [5]:
load_dotenv()
model = ChatGoogleGenerativeAI(model = "gemini-3.8-flash")

In [6]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [11]:
def create_outline(state : BlogState)->BlogState:
    title = state['title']
    # outline = state['outline']

    prompt = f'give outline on topic{title} in 3 sentences'
    outline = model.invoke(prompt).content 
    state['outline'] = outline
    return state

In [12]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state



In [13]:
graph = StateGraph(BlogState)
graph.add_node('create_blog',create_blog)
graph.add_node('create_outline',create_outline)
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)
workflow = graph.compile()

In [14]:
intial_state = {'title': 'Sex Education'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'title': 'Sex Education', 'outline': [{'type': 'text', 'text': '1. **Foundations of Human Biology:** The curriculum begins with essential biological knowledge, covering reproductive anatomy, the physiology of conception, and the physical and hormonal changes of puberty. \n\n2. **Health, Safety, and Relationships:** It progresses to interpersonal dynamics and safety, focusing on mutual consent, effective communication, healthy relationship boundaries, contraception, and the prevention of sexually transmitted infections (STIs). \n\n3. **Empowerment and Decision-Making:** Finally, it addresses psychological and social aspects, emphasizing personal values, critical decision-making, and emotional well-being to foster lifelong responsible and respectful behaviors.', 'extras': {'signature': 'Ep8VCpwVAWkUfRPNqnCv1OfmDTxry94NHcdJiythsu8rZEv9NweYMpmHNw4uJK65PZHg2enJ0YSws65RKchKYNFFQ76qybJMpWOax+AuQAq1P3Ts67EvUkZzLRu/xFqiosW98lMHIEMg4LcPNYCljaKctwiVdodXj330CmoWFA5TgdbneMwbMKFHfgvysBILZX87tbYagM/